In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, Play, jslink, HBox, VBox, HTML, Layout, interactive
from IPython.display import display


# ============================================================
# PERIODIC INPUT AND SUBHARMONIC GENERATION
# ============================================================
#
# This notebook demonstrates the appearance of subharmonic responses
# in a quantized first-order recursive system driven by a periodic
# input.
#
#
# ============================================================
# SYSTEM MODEL
# ============================================================
#
# The unquantized first-order recursive system is
#
#       y[n] = alpha y[n-1] + x[n].
#
# The quantized implementation used in this experiment is
#
#       yq[n] = Q(alpha yq[n-1] + xq[n]),
#
# where
#
#       xq[n] = Q(x[n]).
#
#
# The coefficient alpha itself is not quantized. This allows the
# experiment to focus on the nonlinear effect introduced by state
# quantization.
#
#
# ============================================================
# PERIODIC INPUT
# ============================================================
#
# The input is a periodic impulse train
#
#               A,   n = kN
#       x[n] =
#               0,   otherwise.
#
# Its fundamental period is therefore
#
#       N_in = N.
#
#
# ============================================================
# LINEAR VERSUS QUANTIZED RESPONSE
# ============================================================
#
# For a stable linear system, |alpha| < 1, the steady-state response
# to a periodic input has the same fundamental period as the input.
#
# The quantized recursive system is nonlinear because of Q(.).
#
# Consequently, its steady-state output does not necessarily repeat
# after one input period.
#
# Instead, it may repeat after
#
#       N_out = S N_in
#
# samples, where
#
#       S = 1, 2, 3, ...
#
#
# When
#
#       S > 1,
#
# the output contains a subharmonic response.
#
#
# ============================================================
# SUBHARMONIC ORDER
# ============================================================
#
# The notebook automatically determines the period of the eventual
# quantized response by searching for a repeated augmented state
#
#       ( yq[n], n mod N ).
#
# Including the input phase n mod N is essential:
#
# reaching the same quantized output value at a different phase of
# the periodic forcing does not imply that the future trajectory
# must repeat.
#
# Once the same quantized state AND the same input phase are reached,
# deterministic evolution guarantees repetition.
#
#
# ============================================================
# FIRST-ORDER RESULT ILLUSTRATED HERE
# ============================================================
#
# For the first-order rounding model considered in the accompanying
# theory, subharmonic responses can occur for
#
#       alpha < 0
#
# with an odd input period N.
#
# The important case demonstrated here has
#
#       S = 2,
#
# so that
#
#       N_out = 2 N_in.
#
#
# ============================================================
# DEFAULT EXAMPLE
# ============================================================
#
# The default parameters are
#
#       alpha = -0.90
#       N     = 3
#       K     = 2
#       A     = 0.25
#
# with
#
#       Delta = 2^(-K) = 0.25.
#
# They produce a period-doubled quantized response:
#
#       N_in  = 3
#       N_out = 6
#       S     = 2.
#
#
# ============================================================
# DISPLAY ORGANIZATION
# ============================================================
#
# LEFT TOP
#
#       Periodic input sequence.
#
# LEFT BOTTOM
#
#       Quantized output together with the unquantized reference.
#
# RIGHT TOP
#
#       Period comparison:
#
#           |y[n] - y[n+N]|
#           |y[n] - y[n+2N]|
#
#       For an S = 2 subharmonic, the first quantity remains nonzero
#       while the second becomes zero in the periodic regime.
#
# RIGHT BOTTOM
#
#       Automatic period analyzer and interpretation.
#
#
# ============================================================
# ANIMATION
# ============================================================
#
# The experiment begins at n = 0.
#
# The Play control progressively reveals input and output samples.
#
# During automatic playback, all parameters and the manual time-step
# slider are disabled so that one animation always represents one
# fixed dynamical system.
#
# ============================================================


# ------------------------------------------------------------
# Rounding quantizer: half away from zero
# ------------------------------------------------------------

def round_quantizer(x, Delta):

    x_array = np.asarray(x, dtype=float)

    index = np.where(x_array >= 0.0, np.floor(x_array / Delta + 0.5), np.ceil(x_array / Delta - 0.5))

    result = Delta * index

    if np.ndim(result) == 0:
        return float(result)

    return result


# ------------------------------------------------------------
# Periodic impulse-train input
# ------------------------------------------------------------

def generate_periodic_input(N, amplitude, samples):

    n = np.arange(samples + 1)

    x = np.zeros(samples + 1, dtype=float)

    x[n % N == 0] = amplitude

    return x


# ------------------------------------------------------------
# Unquantized first-order recursive system
# ------------------------------------------------------------

def simulate_unquantized(alpha, x, y0):

    y = np.zeros(len(x), dtype=float)

    y[0] = y0

    for n in range(1, len(x)):

        y[n] = alpha * y[n - 1] + x[n]

    return y


# ------------------------------------------------------------
# Quantized first-order recursive system
# ------------------------------------------------------------

def simulate_quantized(alpha, x, y0, Delta):

    xq = round_quantizer(x, Delta)

    y = np.zeros(len(x), dtype=float)

    y[0] = round_quantizer(y0, Delta)

    for n in range(1, len(x)):

        y[n] = round_quantizer(alpha * y[n - 1] + xq[n], Delta)

    return xq, y


# ------------------------------------------------------------
# Detect eventual period using the augmented state
#
#       (quantized output level, input phase)
#
# Once this pair repeats, all subsequent samples repeat.
# ------------------------------------------------------------

def detect_forced_period(y, N, Delta):

    visited = {}

    for n in range(len(y)):

        level = int(np.round(y[n] / Delta))

        phase = n % N

        key = (level, phase)

        if key in visited:

            first = visited[key]

            second = n

            period = second - first

            return first, second, period

        visited[key] = n

    return None, None, None


# ------------------------------------------------------------
# Fundamental period verification
#
# A detected repeated augmented state always gives a period that is
# an integer multiple of N. The function below checks whether a
# smaller multiple of N also reproduces the periodic tail.
# ------------------------------------------------------------

def refine_output_period(y, start, detected_period, N):

    if start is None or detected_period is None:

        return None


    maximum_multiple = max(1, detected_period // N)


    for S in range(1, maximum_multiple + 1):

        candidate = S * N

        if candidate > detected_period:

            break


        end = min(
            len(y),
            start + 3 * detected_period
        )


        segment = y[start:end]


        if len(segment) <= candidate:

            continue


        if np.allclose(
            segment[candidate:],
            segment[:-candidate],
            atol=1e-12
        ):

            return candidate


    return detected_period


# ------------------------------------------------------------
# Period comparison sequences
# ------------------------------------------------------------

def period_difference(y, shift):

    result = np.full(len(y), np.nan)

    if shift <= 0:

        return result


    for n in range(shift, len(y)):

        result[n] = abs(
            y[n] - y[n - shift]
        )


    return result


# ------------------------------------------------------------
# CSS
# ------------------------------------------------------------

style_html = HTML("""
<style>

.sh-root {
    width: 960px;
    max-width: 960px;
    font-family: Arial, sans-serif;
}

.sh-header {
    background: #303943;
    color: white;
    padding: 8px 14px;
    border-radius: 7px 7px 0 0;
    font-size: 19px;
    font-weight: bold;
}

.sh-intro {
    background: #f5f7f8;
    border: 1px solid #d3d9dd;
    border-top: none;
    padding: 7px 12px;
    border-radius: 0 0 7px 7px;
    font-size: 12px;
    line-height: 1.45;
    margin-bottom: 6px;
}

.sh-accent {
    font-weight: bold;
    color: #3d5563;
}

.sh-controls-title {
    font-size: 12.5px;
    font-weight: bold;
    margin: 0 0 3px 3px;
    color: #303943;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Compact visible documentation
# ------------------------------------------------------------

header_html = HTML("""
<div class="sh-root">

    <div class="sh-header">
        Periodic Input and Subharmonic Generation
    </div>

    <div class="sh-intro">
        <span class="sh-accent">What happens:</span>
        a periodic input of period N drives a quantized recursive system, whose steady-state output may repeat only after S input periods.
        <span class="sh-accent">What to observe:</span>
        when Nout = S Nin with S &gt; 1, a subharmonic response is present; in the demonstrated first-order case, period doubling S = 2 can occur for negative alpha and odd N.
    </div>

</div>
""")


# ============================================================
# MAIN INTERACTIVE FUNCTION
# ============================================================

def plot_subharmonic_lab(alpha=-0.90, N=3, K=2, amplitude=0.25, y0=0.00, samples=60, time_step=0):

    Delta = 2.0**(-K)


    # --------------------------------------------------------
    # Complete input
    # --------------------------------------------------------

    x = generate_periodic_input(
        N,
        amplitude,
        samples
    )


    # --------------------------------------------------------
    # Complete responses
    # --------------------------------------------------------

    y_linear_full = simulate_unquantized(
        alpha,
        x,
        y0
    )


    xq_full, y_quantized_full = simulate_quantized(
        alpha,
        x,
        y0,
        Delta
    )


    # --------------------------------------------------------
    # Full-period analysis
    # --------------------------------------------------------

    period_start_full, period_repeat_full, detected_period_full = detect_forced_period(
        y_quantized_full,
        N,
        Delta
    )


    output_period_full = refine_output_period(
        y_quantized_full,
        period_start_full,
        detected_period_full,
        N
    )


    if output_period_full is not None:

        subharmonic_order_full = output_period_full // N

    else:

        subharmonic_order_full = None


    # --------------------------------------------------------
    # Current animation time
    # --------------------------------------------------------

    current_step = int(
        np.clip(
            time_step,
            0,
            samples
        )
    )


    # --------------------------------------------------------
    # Visible signals
    # --------------------------------------------------------

    n_visible = np.arange(
        current_step + 1
    )


    x_visible = xq_full[:current_step + 1]

    y_linear = y_linear_full[:current_step + 1]

    y_quantized = y_quantized_full[:current_step + 1]


    # --------------------------------------------------------
    # Period detection using only the visible part
    # --------------------------------------------------------

    visible_start, visible_repeat, visible_detected_period = detect_forced_period(
        y_quantized,
        N,
        Delta
    )


    visible_output_period = refine_output_period(
        y_quantized,
        visible_start,
        visible_detected_period,
        N
    )


    if visible_output_period is not None:

        visible_S = visible_output_period // N

    else:

        visible_S = None


    # --------------------------------------------------------
    # Period-comparison sequences
    # --------------------------------------------------------

    difference_N_full = period_difference(
        y_quantized_full,
        N
    )


    difference_2N_full = period_difference(
        y_quantized_full,
        2 * N
    )


    difference_N = difference_N_full[:current_step + 1]

    difference_2N = difference_2N_full[:current_step + 1]


    # ========================================================
    # FIGURE LAYOUT
    # ========================================================

    fig = plt.figure(
        figsize=(13.2, 7.5)
    )


    grid = fig.add_gridspec(
        2,
        2,
        width_ratios=[1.30, 0.90],
        height_ratios=[0.80, 1.15],
        wspace=0.28,
        hspace=0.58
    )


    ax_input = fig.add_subplot(
        grid[0, 0]
    )


    ax_output = fig.add_subplot(
        grid[1, 0]
    )


    ax_difference = fig.add_subplot(
        grid[0, 1]
    )


    ax_monitor = fig.add_subplot(
        grid[1, 1]
    )


    # ========================================================
    # PANEL 1 — PERIODIC INPUT
    # ========================================================

    markerline, stemlines, baseline = ax_input.stem(
        n_visible,
        x_visible,
        basefmt=' '
    )


    plt.setp(
        stemlines,
        color='tab:blue',
        linewidth=1.2
    )


    plt.setp(
        markerline,
        color='tab:blue',
        markerfacecolor='tab:blue',
        markeredgecolor='tab:blue',
        markersize=4.8
    )


    # --------------------------------------------------------
    # Mark input-period boundaries
    # --------------------------------------------------------

    for boundary in range(0, samples + 1, N):

        ax_input.axvline(
            boundary,
            color='0.75',
            linestyle=':',
            linewidth=0.75
        )


    ax_input.set_xlim(
        0,
        samples
    )


    input_max = max(
        abs(np.min(xq_full)),
        abs(np.max(xq_full)),
        Delta
    )


    ax_input.set_ylim(
        -0.15 * input_max,
        1.22 * input_max
    )


    ax_input.set_xlabel(
        'Sample index n',
        fontsize=10.5,
        labelpad=9
    )


    ax_input.set_ylabel(
        r'$x_q[n]$',
        fontsize=11,
        labelpad=7
    )


    ax_input.set_title(
        f'Periodic Input — Fundamental Period N = {N}',
        fontsize=12
    )


    ax_input.tick_params(
        labelsize=9.5
    )


    ax_input.grid(
        True,
        linestyle=':',
        alpha=0.24
    )


    # ========================================================
    # PANEL 2 — OUTPUT
    # ========================================================

    ax_output.plot(
        n_visible,
        y_linear,
        '--',
        color='tab:blue',
        linewidth=1.6,
        label='Unquantized reference'
    )


    markerline, stemlines, baseline = ax_output.stem(
        n_visible,
        y_quantized,
        linefmt='tab:orange',
        markerfmt='o',
        basefmt=' '
    )


    plt.setp(
        stemlines,
        color='tab:orange',
        linewidth=1.15
    )


    plt.setp(
        markerline,
        color='tab:orange',
        markerfacecolor='tab:orange',
        markeredgecolor='tab:orange',
        markersize=4.3
    )


    # --------------------------------------------------------
    # Dummy legend entry for quantized stems
    # --------------------------------------------------------

    ax_output.plot(
        [],
        [],
        'o-',
        color='tab:orange',
        linewidth=1.2,
        markersize=4.3,
        label='Quantized output'
    )


    # --------------------------------------------------------
    # Mark the beginning of the detected periodic regime
    # --------------------------------------------------------

    if period_start_full is not None and current_step >= period_start_full:

        ax_output.axvline(
            period_start_full,
            color='tab:green',
            linestyle=':',
            linewidth=1.5,
            label='Periodic regime begins'
        )


    # --------------------------------------------------------
    # Shade one complete output period after detection
    # --------------------------------------------------------

    if period_start_full is not None and output_period_full is not None and current_step >= period_start_full + output_period_full:

        ax_output.axvspan(
            period_start_full,
            period_start_full + output_period_full,
            alpha=0.07,
            color='tab:green'
        )


    ax_output.axhline(
        0.0,
        color='0.45',
        linewidth=0.7
    )


    ax_output.set_xlim(
        0,
        samples
    )


    output_min = min(
        np.min(y_linear_full),
        np.min(y_quantized_full),
        -Delta
    )


    output_max = max(
        np.max(y_linear_full),
        np.max(y_quantized_full),
        Delta
    )


    output_margin = 0.08 * max(
        output_max - output_min,
        Delta
    )


    ax_output.set_ylim(
        output_min - output_margin,
        output_max + output_margin
    )


    ax_output.set_xlabel(
        'Sample index n',
        fontsize=10.5,
        labelpad=10
    )


    ax_output.set_ylabel(
        r'$y[n]$',
        fontsize=11,
        labelpad=7
    )


    ax_output.set_title(
        'Output Response',
        fontsize=12
    )


    ax_output.tick_params(
        labelsize=9.5
    )


    ax_output.grid(
        True,
        linestyle=':',
        alpha=0.27
    )


    ax_output.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.20),
        ncol=3,
        frameon=False,
        fontsize=9.4,
        columnspacing=1.4,
        handlelength=1.9
    )


    # ========================================================
    # PANEL 3 — PERIOD TEST
    # ========================================================

    ax_difference.plot(
        n_visible,
        difference_N,
        'o-',
        color='tab:orange',
        linewidth=1.4,
        markersize=3.7,
        label=r'$|y[n]-y[n-N]|$'
    )


    ax_difference.plot(
        n_visible,
        difference_2N,
        's-',
        color='tab:green',
        linewidth=1.4,
        markersize=3.7,
        label=r'$|y[n]-y[n-2N]|$'
    )


    ax_difference.axhline(
        0.0,
        color='0.35',
        linestyle=':',
        linewidth=1.0
    )


    ax_difference.set_xlim(
        0,
        samples
    )


    difference_candidates = np.concatenate((
        difference_N_full[np.isfinite(difference_N_full)],
        difference_2N_full[np.isfinite(difference_2N_full)],
        np.array([Delta])
    ))


    difference_max = max(
        np.max(difference_candidates),
        Delta
    )


    ax_difference.set_ylim(
        -0.04 * difference_max,
        1.12 * difference_max
    )


    ax_difference.set_xlabel(
        'Sample index n',
        fontsize=10.5,
        labelpad=10
    )


    ax_difference.set_ylabel(
        'Absolute difference',
        fontsize=10,
        labelpad=7
    )


    ax_difference.set_title(
        'Period Test',
        fontsize=12
    )


    ax_difference.tick_params(
        labelsize=9.5
    )


    ax_difference.grid(
        True,
        linestyle=':',
        alpha=0.27
    )


    ax_difference.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.25),
        ncol=2,
        frameon=False,
        fontsize=9.3,
        columnspacing=1.4
    )


    # ========================================================
    # PANEL 4 — PERIOD ANALYZER
    # ========================================================

    ax_monitor.axis(
        'off'
    )


    # --------------------------------------------------------
    # System information
    # --------------------------------------------------------

    system_text = (
        f'INPUT / SYSTEM\n'
        f'────────────────────────\n'
        f'alpha         : {alpha:+.3f}\n'
        f'|alpha|       : {abs(alpha):.3f}\n'
        f'N input       : {N}\n'
        f'K             : {K}\n'
        f'Delta         : {Delta:.6f}\n'
        f'Amplitude A   : {amplitude:.4f}\n'
        f'Quantized A   : {round_quantizer(amplitude, Delta):.4f}\n'
        f'y[0]          : {round_quantizer(y0, Delta):+.4f}\n'
        f'Time n        : {current_step}'
    )


    # --------------------------------------------------------
    # Current interpretation
    # --------------------------------------------------------

    if visible_output_period is None:

        period_status = (
            'PERIOD NOT YET CONFIRMED\n'
            'Advance the time evolution.'
        )


    elif visible_S == 1:

        period_status = (
            'FUNDAMENTAL RESPONSE\n'
            f'N output = {visible_output_period}\n'
            f'S = {visible_S}\n\n'
            'Output period equals\n'
            'the input period.'
        )


    elif visible_S > 1:

        period_status = (
            'SUBHARMONIC DETECTED\n'
            f'N output = {visible_output_period}\n'
            f'S = {visible_S}\n\n'
            f'Output repeats every\n'
            f'{visible_S} input periods.'
        )


    else:

        period_status = (
            'PERIOD ANALYSIS INCOMPLETE'
        )


    # --------------------------------------------------------
    # Theoretical-condition summary
    # --------------------------------------------------------

    negative_alpha = alpha < 0.0

    odd_N = (N % 2) == 1


    condition_text = (
        f'THEORY CHECK\n'
        f'────────────────────────\n'
        f'alpha < 0      : {"YES" if negative_alpha else "NO"}\n'
        f'N is odd       : {"YES" if odd_N else "NO"}\n\n'
    )


    if negative_alpha and odd_N:

        condition_text += (
            'Conditions associated with\n'
            'period-doubling behavior\n'
            'are present.'
        )


    else:

        condition_text += (
            'The demonstrated S = 2\n'
            'conditions are not both\n'
            'satisfied.'
        )


    # --------------------------------------------------------
    # Full result shown only after sufficient samples have
    # actually become visible.
    # --------------------------------------------------------

    if output_period_full is not None and current_step >= period_repeat_full:

        full_text = (
            f'\n\nCONFIRMED RESULT\n'
            f'────────────────────────\n'
            f'Nin  = {N}\n'
            f'Nout = {output_period_full}\n'
            f'S    = {subharmonic_order_full}'
        )


    else:

        full_text = ''


    result_text = (
        f'PERIOD ANALYZER\n'
        f'────────────────────────\n'
        f'{period_status}\n\n'
        f'{condition_text}'
        f'{full_text}'
    )


    # --------------------------------------------------------
    # Two separated columns
    # --------------------------------------------------------

    ax_monitor.text(
        0.02,
        0.96,
        system_text,
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=9.2,
        family='monospace',
        linespacing=1.37
    )


    ax_monitor.text(
        0.56,
        0.96,
        result_text,
        transform=ax_monitor.transAxes,
        ha='left',
        va='top',
        fontsize=9.2,
        family='monospace',
        linespacing=1.37
    )


    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        'Input–Output Period Analysis in a Quantized Recursive System',
        fontsize=13
    )


    plt.subplots_adjust(
        left=0.06,
        right=0.985,
        top=0.91,
        bottom=0.13
    )


    plt.show()

    plt.close(fig)


# ============================================================
# CONTROLS
# ============================================================


# ------------------------------------------------------------
# Recursive coefficient
# ------------------------------------------------------------

alpha_slider = FloatSlider(
    value=-0.90,
    min=-0.98,
    max=0.98,
    step=0.02,
    description='alpha:',
    continuous_update=True,
    readout_format='.2f',
    style={
        'description_width': '55px'
    },
    layout=Layout(width='350px')
)


# ------------------------------------------------------------
# Input period
# ------------------------------------------------------------

N_slider = IntSlider(
    value=3,
    min=2,
    max=9,
    step=1,
    description='Period N:',
    continuous_update=True,
    style={
        'description_width': '70px'
    },
    layout=Layout(width='300px')
)


# ------------------------------------------------------------
# Quantization precision
# ------------------------------------------------------------

K_slider = IntSlider(
    value=2,
    min=2,
    max=8,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style={
        'description_width': '60px'
    },
    layout=Layout(width='270px')
)


# ------------------------------------------------------------
# Input amplitude
# ------------------------------------------------------------

amplitude_slider = FloatSlider(
    value=0.25,
    min=0.125,
    max=0.75,
    step=0.125,
    description='Amplitude:',
    continuous_update=True,
    readout_format='.3f',
    style={
        'description_width': '75px'
    },
    layout=Layout(width='340px')
)


# ------------------------------------------------------------
# Initial state
# ------------------------------------------------------------

y0_slider = FloatSlider(
    value=0.00,
    min=-0.75,
    max=0.75,
    step=0.125,
    description='y[0]:',
    continuous_update=True,
    readout_format='.3f',
    style={
        'description_width': '55px'
    },
    layout=Layout(width='300px')
)


# ------------------------------------------------------------
# Number of displayed samples
# ------------------------------------------------------------

samples_slider = IntSlider(
    value=60,
    min=30,
    max=120,
    step=5,
    description='Samples:',
    continuous_update=True,
    style={
        'description_width': '65px'
    },
    layout=Layout(width='300px')
)


# ------------------------------------------------------------
# Time-step slider
# ------------------------------------------------------------

time_slider = IntSlider(
    value=0,
    min=0,
    max=60,
    step=1,
    description='Time step n:',
    continuous_update=True,
    style={
        'description_width': '85px'
    },
    layout=Layout(width='670px')
)


# ------------------------------------------------------------
# Play control
# ------------------------------------------------------------

play_control = Play(
    value=0,
    min=0,
    max=60,
    step=1,
    interval=800,
    description='Play',
    disabled=False,
    layout=Layout(width='120px')
)


# ------------------------------------------------------------
# Synchronize Play and Time step
# ------------------------------------------------------------

jslink(
    (play_control, 'value'),
    (time_slider, 'value')
)


# ============================================================
# ANIMATION LOCK
# ============================================================

animation_lock_controls = [
    alpha_slider,
    N_slider,
    K_slider,
    amplitude_slider,
    y0_slider,
    samples_slider,
    time_slider
]


def set_animation_lock(locked):

    for widget in animation_lock_controls:

        widget.disabled = locked


def update_animation_lock(change):

    set_animation_lock(
        bool(change['new'])
    )


# ------------------------------------------------------------
# Detect Play-state trait supported by ipywidgets
# ------------------------------------------------------------

play_traits = play_control.traits()


if 'playing' in play_traits:

    play_state_trait = 'playing'


elif '_playing' in play_traits:

    play_state_trait = '_playing'


else:

    play_state_trait = None


if play_state_trait is not None:

    play_control.observe(
        update_animation_lock,
        names=play_state_trait
    )


set_animation_lock(
    False
)


# ============================================================
# CONTROL SYNCHRONIZATION
# ============================================================


# ------------------------------------------------------------
# Synchronize number of samples and animation range
# ------------------------------------------------------------

def update_time_range(change):

    new_maximum = change['new']

    time_slider.max = new_maximum

    play_control.max = new_maximum


    if time_slider.value > new_maximum:

        time_slider.value = new_maximum


    if play_control.value > new_maximum:

        play_control.value = new_maximum


samples_slider.observe(
    update_time_range,
    names='value'
)


# ------------------------------------------------------------
# Every system-parameter change begins a new experiment
# ------------------------------------------------------------

def restart_time(change):

    time_slider.value = 0

    play_control.value = 0


alpha_slider.observe(
    restart_time,
    names='value'
)


N_slider.observe(
    restart_time,
    names='value'
)


K_slider.observe(
    restart_time,
    names='value'
)


amplitude_slider.observe(
    restart_time,
    names='value'
)


y0_slider.observe(
    restart_time,
    names='value'
)


# ============================================================
# CONTROL LAYOUT
# ============================================================

controls_row_1 = HBox(
    [
        alpha_slider,
        N_slider,
        K_slider
    ],
    layout=Layout(
        width='950px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_row_2 = HBox(
    [
        amplitude_slider,
        y0_slider,
        samples_slider
    ],
    layout=Layout(
        width='940px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_row_3 = HBox(
    [
        play_control,
        time_slider
    ],
    layout=Layout(
        width='830px',
        justify_content='space-between',
        align_items='center'
    )
)


controls_box = VBox(
    [
        HTML("<div class='sh-controls-title'>Period experiment controls</div>"),
        controls_row_1,
        controls_row_2,
        controls_row_3
    ],
    layout=Layout(
        width='960px',
        border='1px solid #d3d9dd',
        padding='6px 8px',
        overflow='visible'
    )
)


# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_subharmonic_lab,
    alpha=alpha_slider,
    N=N_slider,
    K=K_slider,
    amplitude=amplitude_slider,
    y0=y0_slider,
    samples=samples_slider,
    time_step=time_slider
)


plot_output = widget_plot.children[-1]


plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ============================================================
# FINAL NOTEBOOK LAYOUT
# ============================================================

main_layout = VBox(
    [
        header_html,
        controls_box,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ============================================================
# DISPLAY
# ============================================================

display(style_html)

display(main_layout)